# Rekordbox Data Quality Review

In [ ]:
from pathlib import Path
import os

In [ ]:
from dotenv import load_dotenv
environment_file = Path("../env/.env.data_quality")
if not load_dotenv(environment_file, override=True):
    raise ValueError("env file not found...")
print(f"Loaded environment file: {environment_file}")

In [ ]:
import rekordbox_client.dataloading as dataloading

## Read Library XML

In [ ]:
database_folder = os.environ["DATABASE_FOLDER"]

latest = sorted(f for f in os.listdir(database_folder) if f.startswith("rekordbox7_") and f.endswith(".xml"))[-1]

df = dataloading.load_dataframe_from_rekordbox_xml(os.path.join(database_folder, latest))

## Track Duplication check

check whether there are duplicates in the library. It uses the `@Location` key for this check

In [ ]:
results = df["@Location"].duplicated()
duplicate_songnames = []
for index, result in enumerate(results):
    if result:
        duplicate_songnames.append(df.iloc[index]["@Name"])

if len(duplicate_songnames) != 0:
    print(
        f"🚨 {len(duplicate_songnames)} TRACKS WHERE FOUND pointing to the same file on your harddisk:"
    )
    for currsong in duplicate_songnames:
        print("- " + currsong)
else:
    print("✅ No songs appearing more than once were found.")